# Relax structure with MLFF (MACE / UMA / MATTERSIM / NEQUIP) — Machine Learned Force Field

Relax atomic positions locally with ASE using a selectable Machine Learned Force Field (MLFF).


## 1. Set Input Parameters
### 1.1. Interface and Relaxation Parameters


In [ ]:
FOLDER = "uploads"  # local directory containing JSON material files
INTERFACE_NAME = "Interface"  # name of the interface to load from the folder

# MLFF selector
MLFF_NAME = "uma"  # "mace" | "uma" | "mattersim" | "nequip"

# MLFF-specific settings
MLFF_SETTINGS = {
    # MACE settings
    "mace": {
        "model": "large",  # "small" | "medium" | "large"
        "dispersion": True,
        "default_dtype": "float32",
        "device": "cpu",
    },
    # UMA settings
    "uma": {
        "model": "f16",  # "f16" | "int8"
        "task_name": "omat",  # e.g. omat | oc20 | omol
        "device": "cpu",
    },
    # MatterSim settings
    "mattersim": {
        "model": "1m",
        "device": "cpu",
    },
    # NequIP settings
    "nequip": {
        "model": "oam_s",
        "device": "cpu",
    },
}

# Common relaxation settings
RELAXATION_PARAMETERS = {
    "FMAX": 0.05,  # final maximum force on any atom (eV/Å)
}


## 2. Install Packages

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages
from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles

profiles = get_mlff_install_profiles(MLFF_NAME)
await install_packages(profiles)

# PyTorch patches are required in Pyodide for torch-based MLFFs.
if is_pyodide_environment():
    from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

    apply_all_patches(MLFF_NAME)


## 3. Load Materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

interface = load_material_from_folder(FOLDER, INTERFACE_NAME) or Material.create(
    Materials.get_by_name_first_match(INTERFACE_NAME))

### 3.1. Visualize Input Materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize([{"material": interface, "title": interface.name}], viewer=ViewersEnum.wave)
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

## 4. Apply Relaxation
### 4.1. Relax with selected MLFF

In [ ]:
from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

from mat3ra.notebooks_utils.mlff import create_mlff_calculator
from mat3ra.notebooks_utils.ipython.plot._plotly import progress_callback

calculator = create_mlff_calculator(MLFF_NAME, MLFF_SETTINGS[MLFF_NAME])

ase_interface = to_ase(interface)
ase_interface.calc = calculator
dyn = BFGS(ase_interface)

update = progress_callback(
    dynamic_object=dyn,
    value_getter=lambda: float(ase_interface.get_total_energy()),
    value_label="Energy (eV)",
    step_label="Step",
    print_format="Step: {}, Energy: {:.4f} eV",
)
dyn.attach(update, interval=1)
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"])

ase_original_interface = to_ase(interface)
ase_original_interface.calc = calculator
ase_final_interface = ase_interface

original_energy = ase_original_interface.get_total_energy()
relaxed_energy = ase_interface.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")


## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_interface))
material_relaxed = Material.create(from_ase(ase_final_interface))
material_original.name = interface.name
material_relaxed.name = interface.name + f" ({MLFF_NAME.upper()} Relaxed)"

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)


### 5.2. Output interlayer distance before and after relaxation
This requires labels for substrate and film present in the interface structure, which is already done if interface was created with `mat3ra-made`.

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")

### 5.3. Calculate Interface Energy
Interface should have labels marking substrate and film atoms with different tags (e.g. 0 for substrate and 1 for film) for the separation.

In [ ]:
def filter_atoms_by_tag(atoms, material_index):
    return atoms[atoms.get_tags() == material_index]


def calculate_energy(atoms, calc):
    atoms.set_calculator(calc)
    return atoms.get_total_energy()


def calculate_delta_energy(total_energy, *component_energies):
    return total_energy - sum(component_energies)


substrate_original = filter_atoms_by_tag(ase_original_interface, SUBSTRATE_TAG)
layer_original = filter_atoms_by_tag(ase_original_interface, FILM_TAG)
substrate_relaxed = filter_atoms_by_tag(ase_final_interface, SUBSTRATE_TAG)
layer_relaxed = filter_atoms_by_tag(ase_final_interface, FILM_TAG)

original_substrate_energy = calculate_energy(substrate_original, calculator)
original_layer_energy = calculate_energy(layer_original, calculator)
relaxed_substrate_energy = calculate_energy(substrate_relaxed, calculator)
relaxed_layer_energy = calculate_energy(layer_relaxed, calculator)

delta_original = calculate_delta_energy(original_energy, original_substrate_energy, original_layer_energy)
delta_relaxed = calculate_delta_energy(relaxed_energy, relaxed_substrate_energy, relaxed_layer_energy)

area = ase_original_interface.get_volume() / ase_original_interface.cell[2, 2]
n_interface = ase_final_interface.get_global_number_of_atoms()
n_substrate = substrate_relaxed.get_global_number_of_atoms()
n_layer = layer_relaxed.get_global_number_of_atoms()
effective_delta_relaxed = (
                                  relaxed_energy / n_interface
                                  - (relaxed_substrate_energy / n_substrate + relaxed_layer_energy / n_layer)
                          ) / (2 * area)

print(f"Original Substrate energy: {original_substrate_energy:.4f} eV")
print(f"Relaxed Substrate energy:  {relaxed_substrate_energy:.4f} eV")
print(f"Original Layer energy:     {original_layer_energy:.4f} eV")
print(f"Relaxed Layer energy:      {relaxed_layer_energy:.4f} eV")
print("\nDelta between interface energy and sum of component energies")
print(f"Original Delta:            {delta_original:.4f} eV")
print(f"Relaxed Delta:             {delta_relaxed:.4f} eV")
print(f"Original Delta per area:   {delta_original / area:.4f} eV/Ang^2")
print(f"Relaxed Delta per area:    {delta_relaxed / area:.4f} eV/Ang^2")
print(f"Relaxed interface energy:  {relaxed_energy:.4f} eV")
print(
    f"Effective relaxed Delta per area: {effective_delta_relaxed:.4f} eV/Ang^2 ({effective_delta_relaxed / 0.16:.4f} J/m^2)")

## References

[1] mat3ra-made interface builder: https://github.com/Exabyte-io/made  
[2] MACE-MP-0 foundation model: https://github.com/ACEsuit/mace?tab=readme-ov-file#foundation-models  